# NSE Transaction Cost & Market Impact Simulator
## Interactive Walkthrough

This notebook demonstrates every capability of the cost engine — from basic trade cost calculation to market impact estimation with live data.

**Prerequisites:**
```bash
pip install pyyaml requests python-dotenv
```

In [ ]:
from nse_cost_engine import CostEngine, Trade, Segment, TradeType, Side, Exchange

engine = CostEngine(broker="dhan")
print(f"Engine loaded: {engine.broker_name}")

## 1. Basic Trade Cost Calculation

Let's compute the full cost breakdown for buying 100 shares of RELIANCE delivery at ₹2,450.

In [ ]:
trade = Trade(
    symbol="RELIANCE",
    segment=Segment.EQUITY,
    trade_type=TradeType.DELIVERY,
    side=Side.BUY,
    exchange=Exchange.NSE,
    price=2450.0,
    quantity=100,
)

result = engine.calculate(trade)

print(f"Trade Value: ₹{trade.trade_value:,.2f}")
print(f"Total Cost:  ₹{result.total_cost:,.2f}")
print(f"Cost %:      {result.total_cost / trade.trade_value * 100:.4f}%")
print()
print("Component Breakdown:")
for k, v in result.as_dict().items():
    if v > 0 and k not in ('total_regulatory', 'total_cost'):
        print(f"  {k:20s}: ₹{v:>10.2f}")
print(f"  {'TOTAL':20s}: ₹{result.total_cost:>10.2f}")

## 2. Contract Note Rounding

Brokers round STT and stamp duty to the nearest rupee. Our `as_contract_note_dict()` matches what you'll actually be charged.

In [ ]:
cn = result.as_contract_note_dict()
print("Contract Note View (what you actually pay):")
for k, v in cn.items():
    if v > 0:
        print(f"  {k:20s}: ₹{v:>10.2f}")

## 3. Round Trip P&L

Buy at ₹2,000, sell at ₹2,100 — what's the real net P&L after all costs?

In [ ]:
buy = Trade(symbol="RELIANCE", segment=Segment.EQUITY, trade_type=TradeType.DELIVERY,
            side=Side.BUY, exchange=Exchange.NSE, price=2000.0, quantity=100)
sell = Trade(symbol="RELIANCE", segment=Segment.EQUITY, trade_type=TradeType.DELIVERY,
             side=Side.SELL, exchange=Exchange.NSE, price=2100.0, quantity=100)

rt = engine.round_trip(buy, sell)

print(f"Gross P&L:       ₹{rt.gross_pnl:,.2f}")
print(f"Total Costs:     ₹{rt.total_costs:,.2f}")
print(f"Net P&L:         ₹{rt.net_pnl:,.2f}")
print(f"Breakeven Move:  {rt.breakeven_move_pct:.4f}%")
print(f"Cost as % of PnL: {rt.total_costs / rt.gross_pnl * 100:.1f}%")

## 4. Options Trade

Nifty options have a completely different cost profile — STT is on premium, not notional.

In [ ]:
opt = Trade(
    symbol="NIFTY",
    segment=Segment.OPTIONS,
    trade_type=TradeType.INTRADAY,
    side=Side.SELL,
    exchange=Exchange.NSE,
    price=24500.0,
    quantity=1,
    lot_size=75,
    premium=150.0,
    strike_price=24400.0,
)

result = engine.calculate(opt)
print(f"Notional Value: ₹{opt.trade_value:,.2f}")
print(f"Premium Value:  ₹{opt.premium_value:,.2f}  ← costs are based on this")
print(f"Total Cost:     ₹{result.total_cost:,.2f}")
print()
for k, v in result.as_contract_note_dict().items():
    if v > 0:
        print(f"  {k:20s}: ₹{v:>10.2f}")

## 5. MTF Trade with Leverage

MTF trades incur interest on the broker-funded amount + pledge charges. Leverage varies per stock.

In [ ]:
mtf_buy = Trade(
    symbol="RELIANCE",
    segment=Segment.EQUITY,
    trade_type=TradeType.MTF,
    side=Side.BUY,
    exchange=Exchange.NSE,
    price=1322.70,
    quantity=100,
    mtf_leverage=4.55,       # 4.55x leverage
    mtf_holding_days=10,
)

result = engine.calculate(mtf_buy)

print(f"Trade Value:     ₹{mtf_buy.trade_value:,.2f}")
print(f"Your Margin:     ₹{mtf_buy.client_margin:,.2f}")
print(f"Funded by Dhan:  ₹{mtf_buy.funded_amount:,.2f}")
print(f"Leverage:        {mtf_buy.mtf_leverage}×")
print()
print(f"Interest (10d):  ₹{result.mtf_interest:,.2f}")
print(f"Pledge + GST:    ₹{result.mtf_pledge_charges + result.mtf_pledge_gst:,.2f}")
print(f"Other charges:   ₹{result.total_cost - result.mtf_interest - result.mtf_pledge_charges - result.mtf_pledge_gst:,.2f}")
print(f"Total Cost:      ₹{result.total_cost:,.2f}")

## 6. Market Impact — The Hidden Cost

For large orders, market impact dwarfs all regulatory costs. Let's compare across order sizes.

In [ ]:
print(f"{'Shares':>10s}  {'Trade Value':>14s}  {'Regulatory':>12s}  {'Impact':>12s}  {'Total':>12s}  {'Imp/Reg':>8s}")
print("-" * 78)

for qty in [100, 1000, 5000, 10000, 50000, 100000]:
    t = Trade(
        symbol="RELIANCE", segment=Segment.EQUITY,
        trade_type=TradeType.INTRADAY, side=Side.BUY,
        exchange=Exchange.NSE, price=2500.0, quantity=qty,
        daily_volatility=0.017, avg_daily_volume=20000000,
    )
    r = engine.calculate(t)
    reg = r.total_cost_without_impact
    pct = (r.market_impact / reg * 100) if reg > 0 else 0
    print(f"{qty:>10,d}  ₹{t.trade_value:>12,.0f}  ₹{reg:>10,.2f}  ₹{r.market_impact:>10,.2f}  ₹{r.total_cost:>10,.2f}  {pct:>6.0f}%")

## 7. Live Data from Dhan API

Pull real volatility and volume data. Requires `.env` with `DHAN_CLIENT_ID` and `DHAN_ACCESS_TOKEN`.

In [ ]:
try:
    from nse_cost_engine.data_feed import DhanDataFeed, get_security_id
    feed = DhanDataFeed()
    
    if feed.is_configured:
        vol, adv = feed.get_impact_params(get_security_id("RELIANCE"))
        print(f"RELIANCE — Daily Vol: {vol*100:.2f}%, Avg Volume: {adv:,}")
        
        # Now use live data
        trade = Trade(
            symbol="RELIANCE", segment=Segment.EQUITY,
            trade_type=TradeType.INTRADAY, side=Side.BUY,
            exchange=Exchange.NSE, price=2500.0, quantity=10000,
            daily_volatility=vol, avg_daily_volume=adv,
        )
        result = engine.calculate(trade)
        print(f"Market Impact (live data): ₹{result.market_impact:,.2f}")
    else:
        print("Dhan API not configured. Create .env with your credentials.")
except ImportError:
    print("Install requests and python-dotenv for live data.")

## 8. Cross-Broker Comparison

Same trade, different brokers — how do costs compare?

In [ ]:
trade = Trade(
    symbol="NIFTY", segment=Segment.FUTURES,
    trade_type=TradeType.INTRADAY, side=Side.SELL,
    exchange=Exchange.NSE, price=24500.0, quantity=1, lot_size=75,
)

for broker in ["dhan", "zerodha"]:
    eng = CostEngine(broker=broker)
    r = eng.calculate(trade)
    cn = r.as_contract_note_dict()
    print(f"\n{broker.upper()}:")
    for k, v in cn.items():
        if v > 0:
            print(f"  {k:20s}: ₹{v:>10.2f}")

## 9. What-If Analysis

Sensitivity analysis — what if STT on futures was reduced back to 0.02%?

In [ ]:
trade = Trade(
    symbol="NIFTY", segment=Segment.FUTURES,
    trade_type=TradeType.INTRADAY, side=Side.SELL,
    exchange=Exchange.NSE, price=24500.0, quantity=1, lot_size=75,
)

result_current = engine.calculate(trade)
result_old_stt = engine.what_if(trade, stt={"futures": {"rate": 0.0002, "base_on": "trade_value", "side": "sell"}})

print(f"Current STT (0.05%): ₹{result_current.stt:,.2f} → Total: ₹{result_current.total_cost:,.2f}")
print(f"Old STT (0.02%):     ₹{result_old_stt.stt:,.2f} → Total: ₹{result_old_stt.total_cost:,.2f}")
print(f"Savings:             ₹{result_current.total_cost - result_old_stt.total_cost:,.2f}")

## 10. Breakeven Calculator

What's the minimum price move needed to cover all round-trip costs?

In [ ]:
trade = Trade(
    symbol="RELIANCE", segment=Segment.EQUITY,
    trade_type=TradeType.DELIVERY, side=Side.BUY,
    exchange=Exchange.NSE, price=2000.0, quantity=100,
)

be = engine.breakeven(trade)
print(f"Breakeven Move: {be:.4f}%")
print(f"Breakeven Price: ₹{2000 * (1 + be/100):,.2f}")
print(f"Required Move: ₹{2000 * be/100:,.2f} per share")
print(f"\nThis means: any signal predicting < {be:.2f}% move is not worth executing after costs.")